# 🩺 MedVision-AI: Kaggle GPU Cloud Environment & Execution Pipeline

This notebook provides the **Cloud GPU Training Host Environment** for MedVision-AI.
It connects to the source repository at [https://github.com/SwastikPandey1024/MedVision-AI](https://github.com/SwastikPandey1024/MedVision-AI).

> **MLOps Strategy**: Local laptop executes fast CPU smoke tests (`execution_mode: development`). This Kaggle notebook executes full dataset model training and fine-tuning (`execution_mode: full`).

In [ ]:
# Cell 1: Environment & Dynamic Hardware Verification Script
import os
import sys
import platform

# Force Keras 3 TensorFlow Backend
os.environ["KERAS_BACKEND"] = "tensorflow"

import tensorflow as tf
import keras

print("=" * 75)
print("🩺 MedVision-AI Kaggle GPU Environment Report")
print("=" * 75)

# Environment versions
print(f"Python Version       : {platform.python_version()} ({sys.executable})")
print(f"TensorFlow Version   : {tf.__version__}")
print(f"Keras Version        : {keras.__version__}")
print(f"Keras Backend        : {os.environ.get('KERAS_BACKEND', 'tensorflow')}")

# Dynamic GPU Hardware Auto-Detection
gpus = tf.config.list_physical_devices('GPU')
gpu_count = len(gpus)
gpu_available = gpu_count > 0

print(f"\nGPU Availability     : {'YES ✅' if gpu_available else 'NO ❌ (CPU Fallback)'}")
print(f"GPU Device Count     : {gpu_count}")

if gpu_available:
    for i, gpu in enumerate(gpus):
        print(f"\n--- GPU Device #{i+1} Details ---")
        print(f"Device Identifier    : {gpu.name}")
        try:
            details = tf.config.experimental.get_device_details(gpu)
            hardware_name = details.get('device_name', 'NVIDIA GPU')
            print(f"Hardware Model       : {hardware_name}")
            compute_cap = details.get('compute_capability', None)
            if compute_cap:
                print(f"Compute Capability   : {compute_cap}")
        except Exception as err:
            print(f"Hardware Details     : {err}")
        
        # Detect Available VRAM Memory info
        try:
            gpu_mem = tf.config.experimental.get_memory_info(gpu.name)
            current_mb = gpu_mem.get('current', 0) / (1024 * 1024)
            peak_mb = gpu_mem.get('peak', 0) / (1024 * 1024)
            print(f"VRAM Memory Status   : Current: {current_mb:.2f} MB | Peak: {peak_mb:.2f} MB")
        except Exception:
            pass
else:
    print("\n[NOTE] No GPU device discovered in current Kaggle kernel session.")

print("=" * 75)

In [ ]:
# Cell 2: Repository Clone & MedVision Package Installation
!git clone https://github.com/SwastikPandey1024/MedVision-AI.git
%cd MedVision-AI
!pip install -e .

In [ ]:
# Cell 3: Verification of MedVision Package Import & Hardware Config
from medvision.config.settings import load_config
from medvision.utils.device import get_execution_device

config = load_config()
device_info = get_execution_device(config)

print(f"MedVision Config Mode: {config['execution']['mode']}")
print(f"MedVision Active Device: {device_info['device_type']} ({device_info['device_name']})")